# Envoy Proxy — Configuration

Envoy uses a YAML bootstrap file with `listeners`, `clusters`, and an `admin` block.

## Generate an Envoy HTTP proxy config

In [ ]:
import yaml

def envoy_http_config(listen_port, upstream_host, upstream_port, cluster_name="backend"):
    return {
        "static_resources": {
            "listeners": [{
                "name": "listener_0",
                "address": {"socket_address": {"address": "0.0.0.0", "port_value": listen_port}},
                "filter_chains": [{"filters": [{
                    "name": "envoy.filters.network.http_connection_manager",
                    "typed_config": {
                        "@type": "type.googleapis.com/envoy.extensions.filters.network.http_connection_manager.v3.HttpConnectionManager",
                        "stat_prefix": "ingress_http",
                        "http_filters": [{"name": "envoy.filters.http.router",
                                          "typed_config": {"@type": "type.googleapis.com/envoy.extensions.filters.http.router.v3.Router"}}],
                        "route_config": {"name": "local_route", "virtual_hosts": [{
                            "name": "backend", "domains": ["*"],
                            "routes": [{"match": {"prefix": "/"}, "route": {"cluster": cluster_name}}]
                        }]},
                    }
                }]}],
            }],
            "clusters": [{
                "name": cluster_name,
                "type": "STRICT_DNS",
                "connect_timeout": "5s",
                "load_assignment": {"cluster_name": cluster_name, "endpoints": [{
                    "lb_endpoints": [{"endpoint": {"address": {
                        "socket_address": {"address": upstream_host, "port_value": upstream_port}
                    }}}]
                }]},
            }],
        },
        "admin": {"address": {"socket_address": {"address": "0.0.0.0", "port_value": 9901}}},
    }

cfg = envoy_http_config(10000, "httpbin.org", 80)
print(yaml.dump(cfg, default_flow_style=False, sort_keys=False))

## Validate with Envoy --mode validate

In [ ]:
import pathlib, subprocess

cfg_path = pathlib.Path("/tmp/envoy-generated.yaml")
import yaml
cfg_path.write_text(yaml.dump(
    envoy_http_config(10000, "httpbin.org", 80),
    default_flow_style=False, sort_keys=False
))

result = subprocess.run(
    ["docker", "run", "--rm",
     "-v", f"{cfg_path}:/etc/envoy/envoy.yaml:ro",
     "envoyproxy/envoy:v1.29-latest",
     "--mode", "validate", "-c", "/etc/envoy/envoy.yaml"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

## Admin API — useful endpoints

In [ ]:
admin_endpoints = {
    "/clusters":    "Show all clusters and their health",
    "/config_dump": "Full runtime configuration dump",
    "/stats":       "All internal statistics",
    "/listeners":   "Active listeners",
    "/logging":     "Get/set log level (e.g. ?level=debug)",
    "/healthcheck/ok": "Force health-check OK",
}
for path, desc in admin_endpoints.items():
    print(f"curl http://localhost:9901{path:<30}  # {desc}")